## 0. 데이터 전처리 사전 준비

- reviews 폴더에 크롤링된 엑셀 데이터를 저장한다.
- 각 데이터의 주요 컬럼은 다음과 같다.
  - URL: 리뷰가 등록된 가게 URL
  - 가게명: 가게 이름
  - RATING: 별점
  - CONTENT: 리뷰 본문
  - MENU: 주문 메뉴
  - IMAGE_URLS: 리뷰 이미지 URL
  - created_at: 리뷰 생성 시각
  - 지역: 지역

### 목표
- MENU 컬럼에 "리뷰이벤트", "리뷰 이벤트", "리뷰참여" 등의 문구가 포함된 경우 label=1로 설정한다.
- 해당 문구가 없는 경우 label=0으로 설정한다.
- IMAGE_URLS 컬럼을 이용하여 사진 첨부 여부(has_photo)를 생성한다.
- 임베딩을 위한 컬럼 및 원본 컬럼등을 설정한다.

엑셀 파일을 열기 위한 사전 처리
```zsh
uv add openpyxl transformers tokenizers emoji soynlp
```
 - 패키지 필요한 이유
 1. openpyxl	.xlsx 엑셀 파일 읽기
 2. transformers	KcBERT 모델/토크나이저 불러오기
 3. tokenizers	Hugging Face 토크나이저 동작
 4. emoji	이모지 제거/개수 계산
 5. soynlp	ㅋㅋㅋㅋ → ㅋㅋ 반복 문자 정규화

## 1. import
- 라이브러리 및 KCBERT 클린을 위한 임포트

In [11]:
from pathlib import Path
import re
import os
import pandas as pd
import numpy as np
import pandas as pd

import emoji
from soynlp.normalizer import repeat_normalize

import torch
from transformers import AutoTokenizer, AutoModel

## 2. 엑셀 파일 로드 함수 설정 및 로드
- xlsx 및 xls를 한 번에 엽니다.
- 0번 행을 제외하고 1번 부터 읽습니다.
- 그리고 다시 인덱스를 0번 부터 갈 수 있도록 초기화합니다.

In [12]:
def load_excel_files(folder_path, remove_first_row=True):
    folder_path = Path(folder_path)
    excel_files = list(folder_path.rglob("*.xlsx")) + list(folder_path.rglob("*.xls"))

    if len(excel_files) == 0:
        raise FileNotFoundError(f"엑셀 파일을 찾을 수 없습니다: {folder_path}")

    df_list = []

    for file_path in excel_files:
        temp_df = pd.read_excel(file_path)

        if remove_first_row:
            temp_df = temp_df.iloc[1:].reset_index(drop=True)

        temp_df["source_file"] = file_path.name
        df_list.append(temp_df)

    return pd.concat(df_list, ignore_index=True)


df = load_excel_files("reviews", remove_first_row=True)

print("로드 완료:", df.shape)
df.head(3)

로드 완료: (11308, 11)


,URL,NAME,RATING,CONTENT,MENU,DATE,IMAGE_URLS,OWNER_REPLY,created_at,source_file,가게명
0,https://s.baemin.com/pG000flG524ai,철수,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"1개월 전, 가게배달",NaN,"철수님, 저희 기장국수를 찾아와 주셔서 감사합니다 하지만 이번 주문시 늦게 배달되...",2026-05-02 20:28:40 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN
1,https://s.baemin.com/pG000flG524ai,쓰윽쓱,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","2개월 전, 한집배달","[""https://bmreview.cdn.baemin.com/bmreview-qh2...","쓰윽쓱님, 저희 기장국수를 찾아와 주셔서 감사합니다 하지만 이번 주문시 국물이 새어...",2026-05-02 20:28:41 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN
2,https://s.baemin.com/pG000flG524ai,영도남,1.0,NaN,"유부초밥,멸치국수","2개월 전, 알뜰배달",NaN,"영도남님, 저희 기장국수에 다시 찾아와주셔서 감사합니다. 하지만 다소 아쉬움이 있으...",2026-05-02 20:28:41 +0900,배달의민족_리뷰_수집_부산.xlsx,NaN


## 3. 컬럼 정리
- 한글 컬럼 -> 영어로 변환
- 안쓰는 컬럼 삭제

In [13]:
df = df.rename(columns={
    "URL": "store_url",
    "가게명": "store_name",
    "NAME": "reviewer_name",
    "RATING": "rating",
    "CONTENT": "review_text",
    "MENU": "menu_name",
    "DATE": "review_date",
    "IMAGE_URLS": "image_urls",
    "OWNER_REPLY": "owner_reply",
    "created_at": "crawled_at",
    "지역": "region"
})

use_cols = [
    "store_url",
    "store_name",
    "rating",
    "review_text",
    "menu_name",
    "image_urls",
    "source_file"
]

df = df[use_cols]
df.head(3)

,store_url,store_name,rating,review_text,menu_name,image_urls,source_file
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,NaN,"유부초밥,멸치국수",NaN,배달의민족_리뷰_수집_부산.xlsx


## 4. 기본 전처리

1. `review_text_raw`: 원문 리뷰 텍스트 보존
2. `menu_name_raw`: 원문 메뉴명 보존
3. 리뷰 텍스트가 비어 있는 행 제거
4. 별점(`rating`)을 숫자형 데이터로 변환

In [14]:
df["review_text_raw"] = df["review_text"].fillna("").astype(str)
df["menu_name_raw"] = df["menu_name"].fillna("").astype(str)

df = df[df["review_text_raw"].str.strip() != ""].reset_index(drop=True)

df["rating"] = pd.to_numeric(df["rating"], errors="coerce")

df.head(3)

,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥"
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥


## 5. 리뷰 이벤트 여부 판단 후 레이블화
- 메뉴 기준으로 레이블 생성 후 리뷰 라면 

In [15]:
event_menu_pattern = re.compile(
    r"(리뷰\s*이벤트|리뷰이벤트|리뷰\s*참여|리뷰참여|리뷰\s*약속|리뷰약속|review\s*event|ri\s*뷰|니\s*뷰|행복\s*음료|500원의\s*행복|아메리카노/매실/아이스티|rㅣ뷰\s*이벤트|rㅣ뷰\s*eㅣ벤트|포토\s*리뷰|리뷰\s*서비스|리뷰\s*할인|\(리뷰\)|리뷰두\s*찜두|100원의\s*행복|백원의\s*행복)",
    flags=re.IGNORECASE
)

def label_review_event_from_menu(menu):
    menu = str(menu)
    return 1 if event_menu_pattern.search(menu) else 0

df["label"] = df["menu_name_raw"].apply(label_review_event_from_menu)
df.head(3)

,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,0


## 6. KcBERT 입력을 위한 텍스트 정제 및 메타 데이터 생성
- https://github.com/Beomi/KcBERT
- 깃허브 링크 내부의 이모지 관련 체크
- 메타 데이터는 다음과 같다.
    1. 리뷰 텍스트 길이
    2. 이모지 개수
    3. 사진 여부 및 사진 개수


In [16]:
# KcBERT를 위한 전처리
pattern = re.compile(r'[^ .,?!/@$%~％·∼()\x00-\x7F ㄱ-ㅣ가-힣]+')

url_pattern = re.compile(
    r'https?:\/\/(www\.)?[-a-zA-Z0-9@:%._\+~#=]{1,256}'
    r'\.[a-zA-Z0-9()]{1,6}\b'
    r'([-a-zA-Z0-9()@:%_\+.~#?&//=]*)'
)

def clean_review_text(text):
    text = str(text)
    text = url_pattern.sub("", text)
    text = emoji.replace_emoji(text, replace="")
    text = pattern.sub(" ", text)
    text = text.strip()
    text = repeat_normalize(text, num_repeats=2)
    return text

df["cleaned_review_text"] = df["review_text_raw"].apply(clean_review_text)

# 메타 데이터 생성
def has_photo(image_urls):
    if pd.isna(image_urls):
        return 0

    image_urls = str(image_urls).strip()

    if image_urls == "" or image_urls in ["[]", "None", "nan", "NaN"]:
        return 0

    return 1


def count_photos(image_urls):
    if pd.isna(image_urls):
        return 0

    image_urls = str(image_urls).strip()

    if image_urls == "" or image_urls in ["[]", "None", "nan", "NaN"]:
        return 0

    return image_urls.count("http")


df["has_photo"] = df["image_urls"].apply(has_photo)
df["photo_count"] = df["image_urls"].apply(count_photos)

def count_emoji(text):
    return len([char for char in str(text) if char in emoji.EMOJI_DATA])

df["text_length"] = df["review_text_raw"].apply(len)
df["emoji_count"] = df["review_text_raw"].apply(count_emoji)

df.head(3)

,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label,cleaned_review_text,has_photo,photo_count,text_length,emoji_count
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,0,0,16,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0,이걸 수습하기가 참.. 귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러서...,1,2,90,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,0,계란을 깠더니 끈적하고 계란이 그냥 뭉게지네요 썩은 쿤내가 아직도 손에 먹으면 탈 ...,1,3,150,0


## 7. 중복제거
- 크롤링되면서 같은 데이터인 경우를 삭제

In [17]:
df = df.drop_duplicates(
    subset=["store_url", "store_name", "review_text_raw", "menu_name_raw"]
).reset_index(drop=True)

df.head(3)

,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label,cleaned_review_text,has_photo,photo_count,text_length,emoji_count
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,0,0,16,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0,이걸 수습하기가 참.. 귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러서...,1,2,90,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,0,계란을 깠더니 끈적하고 계란이 그냥 뭉게지네요 썩은 쿤내가 아직도 손에 먹으면 탈 ...,1,3,150,0


## 8. 데이터 저장
- 저장
- 작업 폴더 안에 생성됨

In [18]:
df.to_csv("csv/preprocessed_reviews.csv", index=False, encoding="utf-8-sig")
print(os.path.exists("csv/preprocessed_reviews.csv"))

True


## 9. 저장된 CSV 테스트

In [19]:
df_saved = pd.read_csv("csv/preprocessed_reviews.csv", encoding="utf-8-sig")

print("리뷰 개수:", len(df_saved))
print("컬럼 개수:", len(df_saved.columns))
print("데이터 shape:", df_saved.shape)
print("리뷰 이벤트 개수:", (df_saved["label"] == 1).sum())
print("일반 리뷰 개수:", (df_saved["label"] == 0).sum())

df_saved.head(10)

리뷰 개수: 8840
컬럼 개수: 15
데이터 shape: (8840, 15)
리뷰 이벤트 개수: 3150
일반 리뷰 개수: 5690


,store_url,store_name,rating,review_text,menu_name,image_urls,source_file,review_text_raw,menu_name_raw,label,cleaned_review_text,has_photo,photo_count,text_length,emoji_count
0,https://s.baemin.com/pG000flG524ai,NaN,1.0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,NaN,배달의민족_리뷰_수집_부산.xlsx,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,ㄱㅇㄴㅅㄴㄷ이ㅣㄴㄷㄴㅇㄴㅇㄴㅈ,0,0,16,0
1,https://s.baemin.com/pG000flG524ai,NaN,1.0,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,이걸 수습하기가 참..\n귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러...,"멸치국수,비빔국수,비빔밥",0,이걸 수습하기가 참.. 귀찮아서 국수로 간단히 끼니 해결하려 했는데 국물 다 흘러서...,1,2,90,0
2,https://s.baemin.com/pG000flG524ai,NaN,1.0,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,계란을 깠더니 끈적하고 \n계란이 그냥 뭉게지네요\n썩은 쿤내가 아직도 손에…\n먹...,비빔밥,0,계란을 깠더니 끈적하고 계란이 그냥 뭉게지네요 썩은 쿤내가 아직도 손에 먹으면 탈 ...,1,3,150,0
3,https://s.baemin.com/pG000flG524ai,NaN,1.0,국물없이죽이옴.수저안줘서버림.저나하니돈붙여준다함.말안통함.손으로먹나요?별하나도아까움...,떡국,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,국물없이죽이옴.수저안줘서버림.저나하니돈붙여준다함.말안통함.손으로먹나요?별하나도아까움...,떡국,0,국물없이죽이옴.수저안줘서버림.저나하니돈붙여준다함.말안통함.손으로먹나요?별하나도아까움...,1,1,160,0
4,https://s.baemin.com/pG000flG524ai,NaN,1.0,정말 배달전문점에서 완전 싸게 먹는 맛이에요\n다들 맛있다고 들 하시는데 전 이해가...,"소고기국밥,비빔국수",NaN,배달의민족_리뷰_수집_부산.xlsx,정말 배달전문점에서 완전 싸게 먹는 맛이에요\n다들 맛있다고 들 하시는데 전 이해가...,"소고기국밥,비빔국수",0,정말 배달전문점에서 완전 싸게 먹는 맛이에요 다들 맛있다고 들 하시는데 전 이해가 ...,0,0,178,0
5,https://s.baemin.com/pG000flG524ai,NaN,1.0,맛보기수육 6500원짜리 시키니깐 고기 6개 주네요 수육 개당 1000원꼴이네여ㅋㅋ...,떡만두국,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,맛보기수육 6500원짜리 시키니깐 고기 6개 주네요 수육 개당 1000원꼴이네여ㅋㅋ...,떡만두국,0,맛보기수육 6500원짜리 시키니깐 고기 6개 주네요 수육 개당 1000원꼴이네여ㅋㅋ...,1,1,54,0
6,https://s.baemin.com/pG000flG524ai,NaN,3.0,국수맛있어요.소고기국밥도 같이 시켰는데 곱배기로 변경했는데 공기밥이 두개온거는 뭘까...,"소고기국밥,멸치국수","[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,국수맛있어요.소고기국밥도 같이 시켰는데 곱배기로 변경했는데 공기밥이 두개온거는 뭘까...,"소고기국밥,멸치국수",0,국수맛있어요.소고기국밥도 같이 시켰는데 곱배기로 변경했는데 공기밥이 두개온거는 뭘까...,1,2,91,0
7,https://s.baemin.com/pG000flG524ai,NaN,4.0,국물엄청써요..? 원래이런가요 육수용 멸치 오래 우려진건지 반먹다가 포기했습니다..,"멸치국수,유부초밥,비빔밥",NaN,배달의민족_리뷰_수집_부산.xlsx,국물엄청써요..? 원래이런가요 육수용 멸치 오래 우려진건지 반먹다가 포기했습니다..,"멸치국수,유부초밥,비빔밥",0,국물엄청써요..? 원래이런가요 육수용 멸치 오래 우려진건지 반먹다가 포기했습니다..,0,0,46,0
8,https://s.baemin.com/pG000flG524ai,NaN,4.0,최근에 곱빼기 시켜먹었었는데 양 차이가 너무 나네요. 곱빼기가 아니고 보통이 온거 ...,"비빔국수,유부초밥",NaN,배달의민족_리뷰_수집_부산.xlsx,최근에 곱빼기 시켜먹었었는데 양 차이가 너무 나네요. 곱빼기가 아니고 보통이 온거 ...,"비빔국수,유부초밥",0,최근에 곱빼기 시켜먹었었는데 양 차이가 너무 나네요. 곱빼기가 아니고 보통이 온거 ...,0,0,51,0
9,https://s.baemin.com/pG000flG524ai,NaN,4.0,수육 대신 삶은 계란 2개ㅡㅅㅡ;;;\n다들 참고하시고 주문하세요,[1인세트메뉴] 국수/국밥+만두/맛보기수육,"[""https://bmreview.cdn.baemin.com/bmreview-qh2...",배달의민족_리뷰_수집_부산.xlsx,수육 대신 삶은 계란 2개ㅡㅅㅡ;;;\n다들 참고하시고 주문하세요,[1인세트메뉴] 국수/국밥+만두/맛보기수육,0,수육 대신 삶은 계란 2개ㅡㅅㅡ;;; 다들 참고하시고 주문하세요,1,2,35,0


## 10. 분석

In [20]:
# 1. 일반 리뷰와 이벤트 리뷰 분리 (label: 0=일반, 1=이벤트)
normal_df = df_saved[df_saved["label"] == 0]
event_df = df_saved[df_saved["label"] == 1]

# 2. 데이터 수 및 전체 데이터 수 계산
normal_count = len(normal_df)
event_count = len(event_df)
total_count = len(df_saved)

# 3. 데이터프레임으로 결과 표 구성
result_data = {
    "항목": [
        "데이터 수",
        "비율",
        "평균 리뷰 길이",
        "평균 이모지 수",
        "평균 사진 수"
    ],
    "일반 리뷰": [
        f"{normal_count:,}",
        f"{(normal_count / total_count * 100):.1f}%",
        f"{normal_df['text_length'].mean():.1f}",
        f"{normal_df['emoji_count'].mean():.1f}",
        f"{normal_df['photo_count'].mean():.1f}"
    ],
    "이벤트 리뷰": [
        f"{event_count:,}",
        f"{(event_count / total_count * 100):.1f}%",
        f"{event_df['text_length'].mean():.1f}",
        f"{event_df['emoji_count'].mean():.1f}",
        f"{event_df['photo_count'].mean():.1f}"
    ]
}

result_table = pd.DataFrame(result_data)

# 결과 출력
print("=== 리뷰 분석 결과 표 ===")
print(result_table.to_string(index=False))

=== 리뷰 분석 결과 표 ===
      항목 일반 리뷰 이벤트 리뷰
   데이터 수 5,690  3,150
      비율 64.4%  35.6%
평균 리뷰 길이  57.7   54.7
평균 이모지 수   0.7    0.6
 평균 사진 수   0.9    1.0
